# Legal LLM Fine-Tuning — v3 Dataset (QLoRA)

**Dataset:** 1,827 train / 204 validation across 5 tasks:
- Clause drafting (748) — 13 clause types, 6 contract types
- Risk assessment (401) — balanced LOW/MEDIUM/HIGH
- Key terms extraction (255)
- Redline suggestions (275)
- Clause checklist (226)

**Model:** Saul-7B-Instruct (Mistral-based legal LLM) + QLoRA

**Prerequisites:**
1. Open in Colab with **A100 GPU** runtime
2. Set your HuggingFace token in Cell 4
3. Run all cells

**Output:** Fine-tuned LoRA adapter pushed to HuggingFace Hub

## Cell 1 — Verify GPU

In [ ]:
!nvidia-smi
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9 if torch.cuda.is_available() else 0
print(f'\nGPU: {gpu_name} ({vram_gb:.0f} GB)')
assert torch.cuda.is_available(), 'No GPU! Change runtime: Runtime → Change runtime type → A100'
assert vram_gb >= 15, f'Need >= 16GB VRAM, got {vram_gb:.0f}GB. Switch to A100 or T4.'

## Cell 2 — Install dependencies

In [ ]:
!pip install -q torch transformers>=4.40.0 peft>=0.10.0 bitsandbytes>=0.43.0 \
    accelerate>=0.30.0 datasets>=2.19.0 trl>=0.8.0 rouge-score jsonlines pyyaml huggingface_hub

## Cell 3 — Clone repo & load v3 training data

In [ ]:
import os
import json
from pathlib import Path
from collections import Counter

REPO_URL = 'https://github.com/jyoti0512shukla/legal-finetune.git'
REPO_DIR = Path('/content/legal-finetune')

if not REPO_DIR.exists():
    !git clone -b v3 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

DATASET_DIR = REPO_DIR / 'data' / 'training'
OUTPUT_DIR = Path('/content/outputs/saul-7b-legal-lora-v3')

assert DATASET_DIR.exists(), f'Dataset not found at {DATASET_DIR}'

for f in sorted(DATASET_DIR.rglob('*')):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        print(f'  {f.relative_to(DATASET_DIR)} ({size_mb:.1f} MB)')

# Verify v3 dataset
train_count = sum(1 for _ in open(DATASET_DIR / 'train.jsonl'))
val_count = sum(1 for _ in open(DATASET_DIR / 'validation.jsonl'))
print(f'\nv3 Dataset: {train_count} train, {val_count} validation')
assert train_count >= 1800, f'Expected ~1827 train examples, got {train_count} — wrong branch?'

# Show task distribution
tasks = Counter()
with open(DATASET_DIR / 'train.jsonl') as f:
    for line in f:
        row = json.loads(line)
        text = row.get('text', '')
        if 'Assess the risk level' in text: tasks['risk'] += 1
        elif 'Draft a' in text: tasks['drafting'] += 1
        elif 'Check this contract' in text: tasks['checklist'] += 1
        elif 'Extract key terms' in text: tasks['extraction'] += 1
        elif 'needs improvement' in text: tasks['redline'] += 1
for task, count in tasks.most_common():
    print(f'  {task}: {count}')

print(f'\nDataset: {DATASET_DIR}')
print(f'Output:  {OUTPUT_DIR}')

## Cell 4 — HuggingFace login

Get a **write** token from: https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login

HF_TOKEN = 'hf_YOUR_WRITE_TOKEN_HERE'  # <-- REPLACE with your HF write token

# ── HuggingFace Hub repo for the fine-tuned model ──
HF_REPO = 'jyoti0512shuklaorg/saul-legal-v3'  # <-- v3 model repo

assert HF_TOKEN.startswith('hf_') and len(HF_TOKEN) > 10, 'Set your HF token above!'
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN)
print(f'Logged in. Model will be pushed to: https://huggingface.co/{HF_REPO}')

## Cell 5 — Load dataset & show stats

In [ ]:
from datasets import DatasetDict
from collections import Counter

dataset = DatasetDict.load_from_disk(str(DATASET_DIR))

print(f'Train:      {len(dataset["train"]):,} examples')
print(f'Validation: {len(dataset["validation"]):,} examples')
print()

for split in ['train', 'validation']:
    print(f'--- {split} ---')
    types = Counter(dataset[split]['clause_type'])
    sources = Counter(dataset[split]['source'])
    for ct, n in types.most_common():
        print(f'  {ct}: {n}')
    print(f'  Sources: {dict(sources)}')
    print()

print('=== Sample ====') 
print(dataset['train'][0]['text'][:500])

## Cell 6 — Configure training

In [ ]:
import torch

# ── Model ──
BASE_MODEL = 'Equall/Saul-Instruct-v1'
MAX_SEQ_LENGTH = 2048

# ── QLoRA ──
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                   'gate_proj', 'up_proj', 'down_proj']

# ── Training ──
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
WARMUP_STEPS = 10

USE_BF16 = torch.cuda.is_bf16_supported()
print(f'BF16 supported: {USE_BF16}')

total_steps = (len(dataset['train']) * NUM_EPOCHS) // (BATCH_SIZE * GRAD_ACCUM)
print(f'Estimated steps: {total_steps}')
print(f'Estimated time: ~{total_steps * 3 / 60:.0f} min on A100')

## Cell 7 — Load model in 4-bit + attach LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model in 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')
print(f'GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Cell 8 — Train

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    bf16=USE_BF16,
    fp16=not USE_BF16,
    logging_steps=10,
    save_steps=100,
    eval_strategy='steps',
    eval_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    report_to='none',
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
)

print(f'Starting training: {NUM_EPOCHS} epochs, {len(dataset["train"]):,} examples...')
trainer.train()
print('Training complete!')

## Cell 9 — Save adapter

In [ ]:
from pathlib import Path
OUTPUT_DIR = Path('/content/outputs/saul-7b-legal-lora-v3')

adapter_path = OUTPUT_DIR / 'final_adapter'
adapter_path.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))

adapter_size = sum(f.stat().st_size for f in adapter_path.rglob('*') if f.is_file()) / 1e6
print(f'Adapter saved to: {adapter_path}')
print(f'Adapter size: {adapter_size:.0f} MB')

## Cell 10 — Evaluate all 5 tasks

Tests each task the model was trained on: drafting, risk, extraction, checklist, redline.

In [ ]:
import re
import json

EVAL_PROMPTS = [
    # Drafting
    ('drafting', 'Draft a Payment Terms clause for a SaaS Subscription Agreement governed by California, United States law.'),
    ('drafting', 'Draft a Termination clause for a Master Services Agreement governed by New York, United States law.'),
    ('drafting', 'Draft a Force Majeure clause for a Supply Agreement governed by England and Wales law.'),
    # Risk assessment
    ('risk', 'Assess the risk level of this LIABILITY clause:\n\nNeither party shall be liable for any indirect, incidental, special, consequential or punitive damages, regardless of the cause of action or the theory of liability.'),
    ('risk', 'Assess the risk level of this TERMINATION clause:\n\nEither party may terminate this Agreement at any time without cause upon 30 days written notice to the other party.'),
    # Extraction
    ('extraction', 'Extract key terms from this contract:\n\nThis Master Services Agreement ("Agreement") is entered into as of January 15, 2024, by and between Acme Corporation, a Delaware corporation ("Client"), and TechServ Inc., a California corporation ("Provider"). The total contract value shall not exceed $500,000. This Agreement shall be governed by the laws of the State of California. Either party must provide 90 days written notice of termination.'),
    # Checklist
    ('checklist', 'Check this contract for 12 standard clauses:\n\nThis Software License Agreement is entered into between LicenseCo and CustomerCo. The Licensor grants a non-exclusive license to use the Software. Payment of $10,000 annually, due within 30 days of invoice. Either party may terminate with 60 days notice. All disputes shall be resolved by arbitration in New York. The Licensor warrants the Software will perform substantially as described.'),
    # Redline
    ('redline', 'This LIABILITY clause needs improvement. Suggest better language:\n\nThe Provider shall not be liable for any damages whatsoever arising from this Agreement, including but not limited to direct, indirect, or consequential damages, regardless of whether the Provider was advised of the possibility of such damages.'),
]

model.eval()
print('=== V3 MULTI-TASK EVALUATION ===')
print()

results = {'pass': 0, 'fail': 0}

for task, prompt in EVAL_PROMPTS:
    input_text = f'<s>[INST] {prompt} [/INST]'
    inputs = tokenizer(input_text, return_tensors='pt', truncation=True, max_length=2048).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = tokenizer.decode(
        output[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    # Task-specific validation
    status = 'FAIL'
    detail = ''

    if task == 'drafting':
        word_count = len(generated.split())
        sub_clauses = len(re.findall(r'^\d+[.)\s]', generated, re.MULTILINE))
        status = 'PASS' if word_count >= 80 and sub_clauses >= 2 else 'FAIL'
        detail = f'words={word_count}, sub-clauses={sub_clauses}'

    elif task == 'risk':
        try:
            parsed = json.loads(generated)
            has_risk = parsed.get('risk') in ('LOW', 'MEDIUM', 'HIGH')
            has_issues = 'issues' in parsed
            has_summary = 'summary' in parsed
            status = 'PASS' if (has_risk and has_issues and has_summary) else 'FAIL'
            detail = f'risk={parsed.get("risk")}, issues={len(parsed.get("issues", []))}'
        except:
            detail = 'invalid JSON'

    elif task == 'extraction':
        try:
            parsed = json.loads(generated)
            fields = sum(1 for v in parsed.values() if v and v != 'null')
            status = 'PASS' if fields >= 3 else 'FAIL'
            detail = f'fields_extracted={fields}/9'
        except:
            detail = 'invalid JSON'

    elif task == 'checklist':
        try:
            parsed = json.loads(generated)
            clauses = parsed.get('clauses', [])
            status = 'PASS' if len(clauses) >= 8 else 'FAIL'
            detail = f'clauses_checked={len(clauses)}/12'
        except:
            detail = 'invalid JSON'

    elif task == 'redline':
        try:
            parsed = json.loads(generated)
            has_issue = bool(parsed.get('issue'))
            has_suggestion = bool(parsed.get('suggested_language'))
            status = 'PASS' if (has_issue and has_suggestion) else 'FAIL'
            detail = f'issue={has_issue}, suggestion={has_suggestion}'
        except:
            detail = 'invalid JSON'

    results['pass' if status == 'PASS' else 'fail'] += 1
    print(f'[{status}] {task}: {detail}')
    print(f'  Preview: {generated[:200]}...')
    print()

print(f'\n=== Results: {results["pass"]}/{results["pass"]+results["fail"]} passed ===')

## Cell 11 — Merge adapter + push to HuggingFace Hub

Merges LoRA weights into the base model and pushes to your private HF repo.
After this, serve directly on your VM with:
```
HF_TOKEN=hf_... vllm serve jyoti0512shuklaorg/saul-legal-v3 --dtype half --chat-template /tmp/mistral.jinja
```

In [ ]:
import gc
from pathlib import Path

OUTPUT_DIR = Path('/content/outputs/saul-7b-legal-lora-v3')
adapter_path = OUTPUT_DIR / 'final_adapter'

# Free GPU memory from training
del model
del trainer
torch.cuda.empty_cache()
gc.collect()

print('Loading base model in fp16 for merging...')
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)

print('Loading adapter...')
merged_model = PeftModel.from_pretrained(base_model, str(adapter_path))

print('Merging weights...')
merged_model = merged_model.merge_and_unload()

# ── Push to HuggingFace Hub ──
print(f'Pushing merged model to HuggingFace Hub: {HF_REPO} (private)...')
merged_model.push_to_hub(HF_REPO, private=True)
tokenizer.push_to_hub(HF_REPO, private=True)

print()
print('=' * 60)
print('  MODEL PUSHED TO HUGGINGFACE HUB')
print('=' * 60)
print(f'  Repo: https://huggingface.co/{HF_REPO}')
print()
print('  Deploy on your GCP VM:')
print(f'  HF_TOKEN={HF_TOKEN[:10]}... vllm serve {HF_REPO} \\\')
print(f'      --dtype half --chat-template /tmp/mistral.jinja')
print()
print('  .env:')
print(f'  LEGALPARTNER_CHAT_API_URL=http://localhost:8000/v1')
print(f'  LEGALPARTNER_CHAT_API_MODEL={HF_REPO}')
print('=' * 60)